# Phase 3 — RL Molecular Generator


The RL generator uses the trained GNN as a frozen scoring oracle to guide de novo molecule generation. A sequence model proposes candidate SMILES strings, these molecules are evaluated with a reward function that combines predicted binding affinity and basic drug-likeness terms, and reinforcement learning updates the generator so it gradually shifts toward higher-scoring compounds.

The setup here is intentionally modest. I use a character-level LSTM policy with REINFORCE because it is simple to inspect, easy to explain, and good enough for demonstrating how the optimisation loop works end to end.

The workflow in this notebook is:
1. load the frozen GNN oracle from 02,
2. define an EGFR pocket reference,
3. build a SMILES corpus,
4. train a prior model,
5. run RL with a composite reward, and
6. analyse the generated molecules and saved outputs.

## 1. Imports and Configuration

This section sets up the RL environment, cheminformatics helpers, and file paths.

In [ ]:
import sys, os, json, warnings, random, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless backend for Docker
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

import mlflow
import mlflow.pytorch

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, QED, rdMolDescriptors
from rdkit.Chem import rdchem

RDLogger.DisableLog("rdApp.error")

from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import HGTConv
from torch_geometric.nn.aggr import AttentionalAggregation

from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore")
torch.manual_seed(42); random.seed(42); np.random.seed(42)

# ── Paths ─────────────────────────────────────────────────────────────────
ROOT      = Path("..").resolve()
DATA_DIR  = ROOT / "data"
PROCESSED = DATA_DIR / "processed"
CKPT_DIR  = ROOT / "checkpoints"
RL_DIR    = DATA_DIR / "rl_results"
RL_DIR.mkdir(exist_ok=True)

# ── MLflow — local file store fallback when no server env var set ─────────
# Docker sets MLFLOW_TRACKING_URI to the SQL Server backend.
# Local notebook runs use a file store under <repo>/mlruns — no server needed.
MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI",
    str(ROOT / "mlruns"),
)
MLFLOW_EXPERIMENT = "gnn_bind_optimizer"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# ── Dims (must match Phase 1/2) ───────────────────────────────────────────
LIGAND_NODE_DIM  = 28
PROTEIN_NODE_DIM = 26

# ── RL Hyperparameters ────────────────────────────────────────────────────
POLICY_HIDDEN  = 512
POLICY_LAYERS  = 2
EMBED_DIM      = 128
MAX_SMILES_LEN = 120
PRIOR_EPOCHS   = 60       # reduced for Docker CPU runtime
RL_STEPS       = 150
RL_BATCH       = 64       # was 32 — larger batch = more stable REINFORCE gradient
RL_LR          = 3e-4
KL_BETA        = 0.5       # penalty weight for KL(π || π_prior)
EMA_GAMMA      = 0.9       # EMA decay for reward baseline
TEMPERATURE    = 1.0       # sampling temperature

# Reward weights (affinity · QED · SA · MW)
R_WEIGHTS = {"affinity": 0.5, "qed": 0.2, "sa": 0.2, "mw": 0.1}

DEVICE = torch.device("cpu")
print(f"LIGAND_NODE_DIM={LIGAND_NODE_DIM}  PROTEIN_NODE_DIM={PROTEIN_NODE_DIM}")
print(f"Device: {DEVICE}")
print(f"PRIOR_EPOCHS={PRIOR_EPOCHS}  RL_BATCH={RL_BATCH}  RL_STEPS={RL_STEPS}")
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

## 2. Recreate the GNN and Load the Frozen Oracle

The reward function depends on the trained structure-aware scorer, so the first step is to rebuild the model definition and load the saved checkpoint. From this point onward, the GNN is treated as fixed and only used for evaluation.

In [2]:
class HeteroGNN(nn.Module):
    NODE_TYPES = ["ligand", "residue"]
    EDGE_TYPES = [
        ("ligand",  "bond",      "ligand"),
        ("residue", "contact",   "residue"),
        ("residue", "interacts", "ligand"),
        ("ligand",  "interacts", "residue"),
    ]
    META = (NODE_TYPES, EDGE_TYPES)

    def __init__(self, lig_in=LIGAND_NODE_DIM, prot_in=PROTEIN_NODE_DIM,
                 hidden=128, n_layers=4, n_heads=4, dropout=0.1, multitask=True):
        super().__init__()
        self.multitask = multitask
        self.n_layers  = n_layers
        self.lig_proj  = nn.Sequential(nn.Linear(lig_in, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.prot_proj = nn.Sequential(nn.Linear(prot_in, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.convs = nn.ModuleList([
            HGTConv({nt: hidden for nt in self.NODE_TYPES}, hidden, self.META, heads=n_heads)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([
            nn.ModuleDict({nt: nn.LayerNorm(hidden) for nt in self.NODE_TYPES})
            for _ in range(n_layers)
        ])
        self.dropout    = nn.Dropout(dropout)
        self.lig_pool   = AttentionalAggregation(gate_nn=nn.Linear(hidden, 1))
        self.prot_pool  = AttentionalAggregation(gate_nn=nn.Linear(hidden, 1))
        self.trunk = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, hidden // 2), nn.ReLU(),
        )
        head_in = hidden // 2
        self.affinity_head    = nn.Linear(head_in, 1)
        self.pose_head        = nn.Linear(head_in, 1)
        self.selectivity_head = nn.Linear(head_in, 1)

    def forward(self, batch):
        x_dict = {
            "ligand":  self.lig_proj(batch["ligand"].x),
            "residue": self.prot_proj(batch["residue"].x),
        }
        edge_index_dict = {}
        for et in self.EDGE_TYPES:
            try:
                edge_index_dict[et] = batch[et].edge_index
            except (AttributeError, KeyError):
                pass
        for conv, norm_dict in zip(self.convs, self.norms):
            x_new = conv(x_dict, edge_index_dict)
            for nt in self.NODE_TYPES:
                if nt in x_new and x_new[nt] is not None:
                    x_dict[nt] = norm_dict[nt](self.dropout(x_new[nt]) + x_dict[nt])
        lig_emb  = self.lig_pool(x_dict["ligand"],  batch["ligand"].batch)
        prot_emb = self.prot_pool(x_dict["residue"], batch["residue"].batch)
        graph_emb = self.trunk(torch.cat([lig_emb, prot_emb], dim=-1))
        out = {"affinity": self.affinity_head(graph_emb).squeeze(-1)}
        if self.multitask:
            out["pose"]        = self.pose_head(graph_emb).squeeze(-1)
            out["selectivity"] = self.selectivity_head(graph_emb).squeeze(-1)
        return out


# ── Load frozen oracle ────────────────────────────────────────────────────
state_path = ROOT / "src" / "models" / "gnn_state.pt"
assert state_path.exists(), f"GNN state dict not found at {state_path}\nRun Phase 2 first."

saved = torch.load(state_path, map_location=DEVICE, weights_only=False)
hp = saved["hparams"]
oracle = HeteroGNN(
    lig_in=hp["lig_in"], prot_in=hp["prot_in"],
    hidden=hp["hidden"], n_layers=hp["n_layers"],
    n_heads=hp["n_heads"], dropout=hp["dropout"],
    multitask=hp["multitask"],
).to(DEVICE)
oracle.load_state_dict(saved["model_state_dict"])
oracle.eval()
for p in oracle.parameters():
    p.requires_grad_(False)

print(f"Oracle loaded. Test metrics from 02: {saved['test_metrics']}")

Oracle loaded. Test metrics from 02: {'rmse': 1.7021931409835815, 'mae': 1.4917930364608765, 'pearson_r': 0.5786288381605859, 'pose_auc': 0.7962962962962963}


## 3. Reference Protein Pocket and Featurization Helpers

- The reference graph provides a fixed protein pocket context.
- The generated SMILES is converted into a new ligand graph.
- The original ligand from the reference complex is not reused directly.
- Each generated ligand is given a 3D conformer and roughly aligned to the reference binding region.
- The copied residue pocket is then paired with the new ligand.
- Fresh protein-ligand interaction edges are computed before the graph is scored by the frozen GNN.



In [3]:
# ── Featurization (mirrors 01 exactly) ───────────────────────────────
ATOM_TYPES = ["H","C","N","O","F","P","S","Cl","Br","I"]
DEGREES    = [0, 1, 2, 3, 4, 5]
HYBRIDIZE  = [rdchem.HybridizationType.SP, rdchem.HybridizationType.SP2,
              rdchem.HybridizationType.SP3, rdchem.HybridizationType.SP3D,
              rdchem.HybridizationType.SP3D2]
BOND_TYPES = [rdchem.BondType.SINGLE, rdchem.BondType.DOUBLE,
              rdchem.BondType.TRIPLE, rdchem.BondType.AROMATIC]
HBOND_DONORS    = {"N","O"}
HBOND_ACCEPTORS = {"N","O","F"}
PL_EDGE_CUTOFF  = 5.0

def one_hot(val, choices, allow_unknown=True):
    vec = [int(val == c) for c in choices]
    if allow_unknown: vec.append(int(val not in choices))
    return vec

def featurize_ligand_atom(atom):
    return (one_hot(atom.GetSymbol(), ATOM_TYPES)
            + one_hot(atom.GetDegree(), DEGREES)
            + one_hot(atom.GetHybridization(), HYBRIDIZE)
            + [atom.GetFormalCharge(), int(atom.IsInRing()),
               int(atom.GetIsAromatic()), atom.GetTotalNumHs()])

def featurize_ligand_bond(bond):
    return (one_hot(bond.GetBondType(), BOND_TYPES, allow_unknown=False)
            + [int(bond.IsInRing()), int(bond.GetIsAromatic())])


# ── Load reference pocket (highest-pKd training entry) ────────────────────
all_graphs = torch.load(PROCESSED / "dataset.pt", weights_only=False)
ref_graph  = max(all_graphs, key=lambda g: g.y_affinity.item())
ref_pdb_id = ref_graph.pdb_id.replace("_pocket", "")

print(f"Reference pocket: {ref_pdb_id}  pKd={ref_graph.y_affinity.item():.2f}")
print(f"  residue nodes : {ref_graph['residue'].x.shape}")
print(f"  ligand atoms  : {ref_graph['ligand'].x.shape[0]}")

# centroid of reference ligand — used to translate generated conformers
ref_lig_centroid = ref_graph["ligand"].pos.numpy().mean(axis=0)
print(f"  ligand centroid: {ref_lig_centroid.round(2)}")


def smiles_to_hetero_graph(smiles: str, ref_graph: HeteroData,
                            ref_centroid: np.ndarray) -> HeteroData | None:
    """
    Build HeteroData for GNN oracle from a generated SMILES.
    1. Parse + embed 3D conformer (ETKDG)
    2. Translate centroid to match reference ligand (crude pocket alignment)
    3. Copy reference pocket residues + contacts
    4. Compute fresh PL interaction edges at PL_EDGE_CUTOFF
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None
    try:
        mol_h = Chem.AddHs(mol)
        if AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3()) != 0:
            return None
        AllChem.MMFFOptimizeMolecule(mol_h, maxIters=200)
        mol = Chem.RemoveHs(mol_h)
    except Exception:
        return None
    if mol.GetNumConformers() == 0:
        return None

    conf = mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())],
                      dtype=np.float32)
    coords = coords - coords.mean(axis=0) + ref_centroid   # centroid alignment

    x_lig = torch.tensor([featurize_ligand_atom(a) for a in mol.GetAtoms()], dtype=torch.float)
    lig_src, lig_dst, lig_eatts = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ef = featurize_ligand_bond(bond)
        dist = float(np.linalg.norm(coords[i] - coords[j]))
        lig_src += [i, j]; lig_dst += [j, i]
        lig_eatts += [ef + [dist], ef + [dist]]

    ca_coords = ref_graph["residue"].pos.numpy()
    D_pl = cdist(ca_coords, coords)
    pl_sp, pl_dl, pl_ea = [], [], []
    lp_sl, lp_dp, lp_ea2 = [], [], []
    for p_idx, l_idx in zip(*np.where(D_pl <= PL_EDGE_CUTOFF)):
        dist = float(D_pl[p_idx, l_idx])
        la   = mol.GetAtomWithIdx(int(l_idx))
        ef   = [dist, int(la.GetSymbol() in HBOND_DONORS), int(la.GetSymbol() in HBOND_ACCEPTORS)]
        pl_sp.append(int(p_idx)); pl_dl.append(int(l_idx)); pl_ea.append(ef)
        lp_sl.append(int(l_idx)); lp_dp.append(int(p_idx)); lp_ea2.append(ef)

    ei = lambda s, d: (torch.tensor([s, d], dtype=torch.long)
                       if s else torch.zeros((2, 0), dtype=torch.long))
    data = HeteroData()
    data["ligand"].x   = x_lig
    data["ligand"].pos = torch.tensor(coords, dtype=torch.float)
    data["residue"].x  = ref_graph["residue"].x.clone()
    data["residue"].pos = ref_graph["residue"].pos.clone()
    data["ligand","bond","ligand"].edge_index      = ei(lig_src, lig_dst)
    data["ligand","bond","ligand"].edge_attr       = (torch.tensor(lig_eatts, dtype=torch.float)
                                                      if lig_eatts else torch.zeros((0,7)))
    data["residue","contact","residue"].edge_index = ref_graph["residue","contact","residue"].edge_index.clone()
    data["residue","contact","residue"].edge_attr  = ref_graph["residue","contact","residue"].edge_attr.clone()
    data["residue","interacts","ligand"].edge_index = ei(pl_sp, pl_dl)
    data["residue","interacts","ligand"].edge_attr  = (torch.tensor(pl_ea, dtype=torch.float)
                                                       if pl_ea else torch.zeros((0,3)))
    data["ligand","interacts","residue"].edge_index = ei(lp_sl, lp_dp)
    data["ligand","interacts","residue"].edge_attr  = (torch.tensor(lp_ea2, dtype=torch.float)
                                                       if lp_ea2 else torch.zeros((0,3)))
    data.y_affinity = torch.tensor([0.0], dtype=torch.float)
    data.y_pose     = torch.tensor([0.0], dtype=torch.float)
    data.y_select   = torch.tensor([0.0], dtype=torch.float)
    return data


# Smoke test
_test_smi = "CCc1nn(C)c2ccc(Nc3ncnc4[nH]ccc34)cc12"  # ruxolitinib-like
_test_g   = smiles_to_hetero_graph(_test_smi, ref_graph, ref_lig_centroid)
print(f"\nOracle smoke test on ruxolitinib-like:")
print(f"  graph built   : {_test_g is not None}")
if _test_g is not None:
    _batch = PyGLoader([_test_g], batch_size=1)
    with torch.no_grad():
        _out = oracle(next(iter(_batch)).to(DEVICE))
    print(f"  oracle pKd    : {_out['affinity'].item():.3f}")

Reference pocket: 6e9a  pKd=11.92
  residue nodes : torch.Size([43, 26])
  ligand atoms  : 40
  ligand centroid: [16.22 22.6  17.05]

Oracle smoke test on ruxolitinib-like:
  graph built   : True
  oracle pKd    : 7.556


## 4. Build the SMILES Corpus

This cell builds the SMILES corpus used to train the initial sequence-model prior. It combines ligands extracted from the PDBBind training set with a small set of known `EGFR` or kinase inhibitor molecules, so the generator starts from chemically plausible structures that are both drug-like and more relevant to the intended target class.

The prior is trained mainly on general PDBBind-derived ligands, with a small set of EGFR or kinase inhibitor examples added as a weak steering signal. As a result, the model is not truly EGFR-specific, but its initial sampling distribution is nudged toward more kinase-relevant chemistry.

In [4]:
# Known EGFR/kinase inhibitors — seed the prior toward drug-like chemical space
EGFR_INHIBITORS = [
    "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1",           # erlotinib
    "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OC",                  # gefitinib
    "CS(=O)(=O)CCNCc1ccc(Oc2ncnc3cc(Cl)ccc23)cc1",           # lapatinib-like
    "C=CC(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OC",           # afatinib
    "COc1cc2c(Nc3cccc(Cl)c3F)ncnc2cc1NC(=O)/C=C/CN(C)C",     # dacomitinib
    "C=CC(=O)Nc1cccc(N2C(=O)c3ccccc3C2=O)c1",
    "Cc1ccc(Nc2nccc(-c3cccnc3)n2)cc1",                        # crizotinib-like
    "O=C(c1ccc(F)cc1)c1ccc(NC(=O)c2cccc(C(F)(F)F)c2)cc1",
    "Cc1ccc(-c2cc(NC(=O)c3ccc(CN4CCN(C)CC4)cc3)ncn2)cc1",
    "CN1CCN(c2ccc(Nc3ncnc4ccc(OC)cc34)cc2)CC1",               # imatinib-like
    "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",
    "O=C(/C=C/c1ccccc1)Nc1cc2c(Nc3ccccc3Cl)ncnc2cc1",
    "Fc1cc2c(Nc3ccnc(Nc4ccccc4)n3)ncnc2cc1",
    "CC(C)c1nc(N2CCC(N)CC2)ncc1-c1cnn2ncccc12",
    "CN(Cc1cnc2ccccc2n1)C(=O)c1ccc(Cl)s1",
    "Cc1nc2ccccc2n1CC(=O)Nc1ccc(OC(F)(F)F)cc1",
    "O=C(Nc1ccc(Oc2ccnc3[nH]ccc23)cc1)c1ccc(Cl)cc1F",        # erlotinib scaffold
    "COc1ccc(Nc2ncnc3ccc(F)cc23)cc1NC(=O)/C=C/C(C)=O",
    "Cc1ccc2ncnc(Nc3ccc(OCc4cccc(F)c4)c(Cl)c3)c2c1",
    "O=C(Nc1cc2ncnc(Nc3cccc(Br)c3)c2cc1)c1ccco1",
    # generic drug-like scaffolds
    "c1ccc2ncccc2c1", "c1ccc(-c2ccccn2)nc1", "c1cnc2ccccc2c1",
    "Cc1ccc(C(=O)O)cc1", "CC(=O)Nc1ccc(O)cc1",
    "c1ccc(NC(=O)c2cccc(C(F)(F)F)c2)cc1",
    "CN1CCNCC1", "C1COCCN1", "c1ccc(N2CCCC2)cc1",
]

# Extract SMILES from PDBbind training set
def sdf_to_smiles(sdf_path):
    suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=True, sanitize=False)
    for mol in suppl:
        if mol is None: continue
        try:
            Chem.SanitizeMol(mol, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE)
            mol.UpdatePropertyCache(strict=False)
            smi = Chem.MolToSmiles(mol)
            if smi: return smi
        except Exception:
            pass
    return None

REFINED_DIR = DATA_DIR / "pdbbind" / "refined-set"
training_smiles = []
for g in all_graphs:
    pdb_id = g.pdb_id.replace("_pocket", "")
    sdf = REFINED_DIR / pdb_id / f"{pdb_id}_ligand.sdf"
    if sdf.exists():
        s = sdf_to_smiles(sdf)
        if s:
            training_smiles.append(s)

corpus = list(set(EGFR_INHIBITORS + training_smiles))
# Canonicalize + filter
valid_corpus = []
for s in corpus:
    m = Chem.MolFromSmiles(s)
    if m and 5 <= m.GetNumAtoms() <= 80:
        valid_corpus.append(Chem.MolToSmiles(m))
valid_corpus = list(set(valid_corpus))

print(f"Training SMILES extracted : {len(training_smiles)}")
print(f"EGFR inhibitors           : {len(EGFR_INHIBITORS)}")
print(f"Total corpus (deduped)    : {len(valid_corpus)}")
print(f"Sample: {valid_corpus[0][:60]}")

Training SMILES extracted : 150
EGFR inhibitors           : 29
Total corpus (deduped)    : 160
Sample: CCCN(CCC)C(=O)c1cc(C)cc(C(=O)N[C@@H](Cc2cc(F)cc(F)c2)[C@H](O


[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:16:44] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

## 5. Character-Level Tokenizer

This cell defines the tokenizer used by the sequence model. It converts SMILES strings into token IDs for training, and supports decoding generated token sequences back into valid SMILES strings.


In [5]:
class SMILESTokenizer:
    """
    Character-level tokenizer for SMILES strings.
    Two-character tokens (Cl, Br, Si, Se, ...) are extracted before single chars.
    """
    TWO_CHAR = {"Cl", "Br", "Si", "Se", "Na", "Li", "Mg", "Ca", "Fe", "Cu",
                "Zn", "Al", "As", "@@"}
    PAD, BOS, EOS, UNK = "<PAD>", "<BOS>", "<EOS>", "<UNK>"

    def __init__(self, smiles_list: list[str]):
        chars: set[str] = set()
        for s in smiles_list:
            i = 0
            while i < len(s):
                two = s[i:i+2]
                if two in self.TWO_CHAR:
                    chars.add(two); i += 2
                else:
                    chars.add(s[i]); i += 1
        self.vocab = [self.PAD, self.BOS, self.EOS, self.UNK] + sorted(chars)
        self.c2i = {c: idx for idx, c in enumerate(self.vocab)}
        self.i2c = {idx: c for idx, c in enumerate(self.vocab)}
        self.pad_idx = self.c2i[self.PAD]
        self.bos_idx = self.c2i[self.BOS]
        self.eos_idx = self.c2i[self.EOS]
        self.unk_idx = self.c2i[self.UNK]

    def tokenize(self, s: str) -> list[str]:
        tokens, i = [], 0
        while i < len(s):
            two = s[i:i+2]
            if two in self.TWO_CHAR:
                tokens.append(two); i += 2
            else:
                tokens.append(s[i]); i += 1
        return tokens

    def encode(self, s: str, add_bos=True, add_eos=True) -> list[int]:
        ids = [self.c2i.get(t, self.unk_idx) for t in self.tokenize(s)]
        if add_bos: ids = [self.bos_idx] + ids
        if add_eos: ids = ids + [self.eos_idx]
        return ids

    def decode(self, ids: list[int]) -> str:
        chars = []
        for i in ids:
            if i == self.eos_idx: break
            if i in (self.pad_idx, self.bos_idx): continue
            chars.append(self.i2c.get(i, "?"))
        return "".join(chars)

    def __len__(self) -> int:
        return len(self.vocab)


tokenizer = SMILESTokenizer(valid_corpus)
print(f"Vocabulary size : {len(tokenizer)}")
print(f"Tokens          : {tokenizer.vocab[:10]}  …")

# Verify round-trip
_smi = valid_corpus[0]
_enc = tokenizer.encode(_smi)
_dec = tokenizer.decode(_enc)
assert _dec == _smi, f"Round-trip mismatch:\n  in : {_smi}\n  out: {_dec}"
print(f"Round-trip OK on '{_smi[:40]}'")

Vocabulary size : 37
Tokens          : ['<PAD>', '<BOS>', '<EOS>', '<UNK>', '#', '(', ')', '+', '-', '/']  …
Round-trip OK on 'CCCN(CCC)C(=O)c1cc(C)cc(C(=O)N[C@@H](Cc2'


## 6. LSTM Policy and Prior Training
![Model architecture](../Figs/LSTM-model-arch.png)

In [6]:
class SMILESPolicy(nn.Module):
    """
    Character-level LSTM SMILES generator.
    forward(): teacher-forcing logits for training.
    sample():  autoregressive sampling for RL.
    """
    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, n_layers: int, dropout: float = 0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers   = n_layers
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, n_layers,
                             batch_first=True,
                             dropout=dropout if n_layers > 1 else 0.0)
        self.fc    = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor, hidden=None):
        out, hidden = self.lstm(self.embed(x), hidden)
        return self.fc(out), hidden

    def init_hidden(self, batch_size: int, device):
        z = torch.zeros(self.n_layers, batch_size, self.hidden_dim, device=device)
        return z, z.clone()

    @torch.no_grad()
    def sample(self, tokenizer, n: int, max_len: int = MAX_SMILES_LEN,
               temperature: float = TEMPERATURE, device=DEVICE):
        """Autoregressive sampling — returns (smiles_list, log_prob_sums)."""
        self.eval()
        tokens = torch.full((n, 1), tokenizer.bos_idx, dtype=torch.long, device=device)
        hidden = self.init_hidden(n, device)
        done   = torch.zeros(n, dtype=torch.bool, device=device)
        seqs   = []
        lp_sum = torch.zeros(n, device=device)

        for _ in range(max_len):
            logits, hidden = self.forward(tokens, hidden)
            logits = logits[:, -1, :] / temperature      # [n, vocab]
            log_p  = F.log_softmax(logits, dim=-1)
            next_t = torch.multinomial(log_p.exp(), 1)   # [n, 1]
            lp_sum += log_p.gather(1, next_t).squeeze(1) * (~done).float()
            done  |= next_t.squeeze(1) == tokenizer.eos_idx
            seqs.append(next_t)
            tokens = next_t
            if done.all():
                break

        seq_tensor = torch.cat(seqs, dim=1)              # [n, seq]
        smiles = [tokenizer.decode(seq_tensor[i].tolist()) for i in range(n)]
        return smiles, lp_sum

    def log_prob(self, token_ids: torch.Tensor, device=DEVICE) -> torch.Tensor:
        """
        Compute sum of log P(token | prefix) for a sequence.
        token_ids: [seq_len]  (includes BOS, excludes padding after EOS)
        Returns scalar. Does NOT mutate model train/eval state.
        """
        # use no_grad context — caller is responsible for gradient tracking
        x   = token_ids[:-1].unsqueeze(0).to(device)   # [1, T-1]
        tgt = token_ids[1:].unsqueeze(0).to(device)     # [1, T-1]
        logits, _ = self.forward(x)                     # [1, T-1, vocab]
        log_p = F.log_softmax(logits, dim=-1)
        return log_p.squeeze(0).gather(1, tgt.squeeze(0).unsqueeze(1)).sum()


# ── Build dataset for prior training ─────────────────────────────────────
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, tokenizer, max_len):
        self.data = []
        for s in smiles_list:
            ids = tokenizer.encode(s)
            if len(ids) <= max_len:
                self.data.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]


def collate_pad(batch):
    max_len = max(b.size(0) for b in batch)
    padded  = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, b in enumerate(batch):
        padded[i, :b.size(0)] = b
    return padded


smiles_ds     = SMILESDataset(valid_corpus, tokenizer, MAX_SMILES_LEN)
smiles_loader = DataLoader(smiles_ds, batch_size=32, shuffle=True, collate_fn=collate_pad)
print(f"Prior training sequences: {len(smiles_ds)}")

# ── Initialise policy ─────────────────────────────────────────────────────
policy = SMILESPolicy(len(tokenizer), EMBED_DIM, POLICY_HIDDEN, POLICY_LAYERS).to(DEVICE)
prior  = None   # will be a frozen copy after prior training

optimizer_prior = Adam(policy.parameters(), lr=1e-3)

# ── Prior training loop (teacher forcing) ─────────────────────────────────
prior_losses = []
print(f"Training LSTM prior for {PRIOR_EPOCHS} epochs …")
for epoch in tqdm(range(PRIOR_EPOCHS)):
    policy.train()
    epoch_loss = 0.0
    for batch in smiles_loader:
        batch = batch.to(DEVICE)
        x   = batch[:, :-1]   # input: BOS … last-1
        tgt = batch[:, 1:]    # target: second … EOS
        logits, _ = policy(x)
        loss = F.cross_entropy(
            logits.reshape(-1, len(tokenizer)),
            tgt.reshape(-1),
            ignore_index=0,   # pad
        )
        optimizer_prior.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer_prior.step()
        epoch_loss += loss.item()
    prior_losses.append(epoch_loss / len(smiles_loader))

# Freeze a copy as the reference prior
prior = copy.deepcopy(policy).eval()
for p in prior.parameters():
    p.requires_grad_(False)

print(f"Prior training done. Final loss: {prior_losses[-1]:.4f}")

# Sample with eval mode to check true prior quality (no dropout)
policy.eval()
_samp, _ = policy.sample(tokenizer, n=20, device=DEVICE)
valid_n  = sum(1 for s in _samp if Chem.MolFromSmiles(s) is not None)
print(f"Prior sample validity (eval, n=20): {valid_n}/20")
for s in _samp[:5]:
    print(f"  {s[:60]}")

Prior training sequences: 154
Training LSTM prior for 60 epochs …


  0%|          | 0/60 [00:00<?, ?it/s]

Prior training done. Final loss: 0.1611


Prior sample validity (eval, n=20): 4/20
  [NH3+]C[C@H](C[C@H]1C[C@H](C(=O)O)N=[N]1c5cccc(F)1)c1cccc(C(
  COc1ccc(Nc2nccn([C@@H]3O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)[nH]1
  C[C@H](O)[C@@H](CCc1ccccc1)[C@@H](C)CC[C@@H]2
  O=C(NSC(=O)c1cccc1C[C@H]1[C@@H]2NC(=O)O
  c1ccc(C(=O)O)cc1


## 7. Reward Function

LSTM learns to generate plausible SMILES, but that alone does not tell it which molecules are better. The reward function is what adds that preference.

The reward function assigns a scalar score to each generated molecule by combining predicted binding affinity from the frozen GNN with auxiliary chemistry-based terms such as QED, synthetic accessibility, and molecular-weight preference. 

The reward function is:

`R = 0.5 * r_affinity + 0.2 * r_qed + 0.2 * r_sa + 0.1 * r_mw`

In [7]:
# SA score — try rdkit.Contrib first, fall back to ring-count proxy
def _sa_proxy_score(mol: Chem.Mol) -> float:
    # Proxy: heavier penalty for many rings + stereocentres
    n_rings  = mol.GetRingInfo().NumRings()
    n_stereo = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    return min(10.0, 1.0 + n_rings * 0.4 + n_stereo * 0.3)

try:
    from rdkit.Contrib.SA_Score import sascorer as _sascorer
    def sa_score(mol: Chem.Mol) -> float:
        try:
            return _sascorer.calculateScore(mol)
        except Exception:
            return _sa_proxy_score(mol)
except Exception:
    def sa_score(mol: Chem.Mol) -> float:
        return _sa_proxy_score(mol)


def compute_reward(smiles: str) -> dict:
    """
    Compute the 4-component reward for a generated SMILES.

    Returns dict with:
      r_affinity, r_qed, r_sa, r_mw, reward_total
      (invalid SMILES → all zeros)
    """
    result = {"r_affinity": 0.0, "r_qed": 0.0, "r_sa": 0.0,
              "r_mw": 0.0, "reward_total": 0.0, "valid": False,
              "pred_pkd": float("nan")}

    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return result

    result["valid"] = True

    # ── Ligand-based components ────────────────────────────────────────
    qed_val = QED.qed(mol)
    sa_val  = sa_score(mol)
    mw      = Descriptors.MolWt(mol)

    result["r_qed"] = float(qed_val)                   # [0, 1]
    result["r_sa"]  = float(max(0.0, 1.0 - sa_val / 10.0))  # [0, 1]
    result["r_mw"]  = float(1.0 if mw <= 500 else max(0.0, 1.0 - (mw - 500) / 200))

    # ── GNN affinity oracle ────────────────────────────────────────────
    graph = smiles_to_hetero_graph(smiles, ref_graph, ref_lig_centroid)
    if graph is not None:
        loader = PyGLoader([graph], batch_size=1)
        with torch.no_grad():
            pred = oracle(next(iter(loader)).to(DEVICE))
        pkd = float(pred["affinity"].item())
        result["pred_pkd"] = pkd
        # Normalise to [0, 1]: pKd ∈ [2, 12] → clamp + scale
        result["r_affinity"] = float(np.clip((pkd - 2.0) / 10.0, 0.0, 1.0))

    # ── Weighted sum ──────────────────────────────────────────────────
    result["reward_total"] = (
        R_WEIGHTS["affinity"] * result["r_affinity"]
        + R_WEIGHTS["qed"]    * result["r_qed"]
        + R_WEIGHTS["sa"]     * result["r_sa"]
        + R_WEIGHTS["mw"]     * result["r_mw"]
    )
    return result


# Test on erlotinib
_rew = compute_reward("C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1")
print("Erlotinib reward:")
for k, v in _rew.items():
    print(f"  {k:15s}: {v}")


Erlotinib reward:
  r_affinity     : 0.5430843830108643
  r_qed          : 0.4178840975839869
  r_sa           : 0.7521698265291962
  r_mw           : 1.0
  reward_total   : 0.6055529763280688
  valid          : True
  pred_pkd       : 7.430843830108643


[11:21:29] DEPRECATION WARNING: please use MorganGenerator


## 8. REINFORCE Training Loop

Here the policy is updated with policy gradients while a KL-style regularization term keeps it from collapsing too far away from the prior. The objective is not only to increase reward, but to do so without destroying validity and diversity.

In [ ]:
def sample_with_grad(policy, tokenizer, n, max_len, device, temperature=1.0):
    """
    Autoregressive SMILES sampling WITH gradient flow through log-probs.

    Policy runs in EVAL mode (dropout disabled) so token distributions
    are deterministic given hidden state — matching log_prob() evaluation.
    Gradients still flow through the log-prob gather because no torch.no_grad()
    wraps the forward pass here.

    Returns: smiles list, total_log_probs [n] (requires_grad=True)
    """
    policy.eval()   # CRITICAL: eval disables LSTM dropout during generation
                    # train() here compounded dropout over 120 steps → 0% validity
    tokens   = torch.full((n, 1), tokenizer.bos_idx, dtype=torch.long, device=device)
    h        = torch.zeros(policy.n_layers, n, policy.hidden_dim, device=device)
    c        = torch.zeros(policy.n_layers, n, policy.hidden_dim, device=device)
    done     = torch.zeros(n, dtype=torch.bool, device=device)
    step_lps = []
    sampled  = []

    for _ in range(max_len):
        logits, (h, c) = policy.lstm(policy.embed(tokens), (h, c))
        logits = policy.fc(logits[:, -1, :]) / temperature     # [n, vocab]
        log_p  = F.log_softmax(logits, dim=-1)                 # [n, vocab]

        with torch.no_grad():
            next_t = torch.multinomial(log_p.exp().clamp(min=1e-9), 1)  # [n,1]

        step_lp = log_p.gather(1, next_t).squeeze(1)          # [n], has grad
        step_lps.append(step_lp * (~done).float())

        done  |= next_t.squeeze(1) == tokenizer.eos_idx
        sampled.append(next_t)
        tokens = next_t
        if done.all():
            break

    seq_tensor    = torch.cat(sampled, dim=1)                  # [n, T]
    total_log_p   = torch.stack(step_lps, dim=1).sum(dim=1)   # [n]
    smiles_list   = [tokenizer.decode(seq_tensor[i].tolist()) for i in range(n)]
    return smiles_list, total_log_p


optimizer_rl = Adam(policy.parameters(), lr=RL_LR)

# ── RL bookkeeping ────────────────────────────────────────────────────────
history = {
    "step": [], "reward_mean": [], "reward_max": [],
    "validity": [], "kl_penalty": [], "aug_reward_mean": [],
}
all_molecules = []   # list of {step, smiles, reward_total, pred_pkd, ...}
baseline = 0.0       # EMA reward baseline

print(f"Starting REINFORCE: {RL_STEPS} steps × {RL_BATCH} molecules/step")
print(f"KL_BETA={KL_BETA}  EMA_GAMMA={EMA_GAMMA}")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
with mlflow.start_run(run_name="reinforce_rl") as rl_run:
    # Log all RL hyperparameters up front
    mlflow.log_params({
        "rl_steps": RL_STEPS, "rl_batch": RL_BATCH, "rl_lr": RL_LR,
        "kl_beta": KL_BETA, "ema_gamma": EMA_GAMMA,
        "prior_epochs": PRIOR_EPOCHS, "temperature": TEMPERATURE,
        "policy_hidden": POLICY_HIDDEN, "policy_layers": POLICY_LAYERS,
        "embed_dim": EMBED_DIM, "max_smiles_len": MAX_SMILES_LEN,
        "r_weight_affinity": R_WEIGHTS["affinity"],
        "r_weight_qed": R_WEIGHTS["qed"],
        "r_weight_sa": R_WEIGHTS["sa"],
        "r_weight_mw": R_WEIGHTS["mw"],
    })

    for step in tqdm(range(RL_STEPS)):
        # ── Sample batch — eval mode, gradients flow through log_p ──────
        smiles_list, total_lp = sample_with_grad(
            policy, tokenizer, RL_BATCH, MAX_SMILES_LEN, DEVICE
        )

        # ── Compute rewards (no grad) ────────────────────────────────────
        rewards_raw = np.array([compute_reward(s)["reward_total"] for s in smiles_list])

        # ── KL penalty vs frozen prior ───────────────────────────────────
        kl_penalties = np.zeros(RL_BATCH)
        for i, s in enumerate(smiles_list):
            mol = Chem.MolFromSmiles(s)
            if mol is None:
                continue
            ids = tokenizer.encode(s)
            if len(ids) < 2:
                continue
            id_t = torch.tensor(ids, dtype=torch.long, device=DEVICE)
            with torch.no_grad():
                lp_policy = policy.log_prob(id_t, DEVICE).item()
                lp_prior  = prior.log_prob(id_t, DEVICE).item()
            kl_penalties[i] = lp_policy - lp_prior

        aug_rewards = rewards_raw - KL_BETA * kl_penalties
        aug_t = torch.tensor(aug_rewards, dtype=torch.float32, device=DEVICE)

        # ── REINFORCE with EMA baseline ──────────────────────────────────
        baseline = EMA_GAMMA * baseline + (1 - EMA_GAMMA) * float(aug_t.mean())
        advantage = aug_t - baseline

        loss = -(advantage.detach() * total_lp).mean()

        optimizer_rl.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer_rl.step()

        # ── Per-step metrics ──────────────────────────────────────────────
        validity = sum(1 for s in smiles_list if Chem.MolFromSmiles(s)) / RL_BATCH
        history["step"].append(step)
        history["reward_mean"].append(float(rewards_raw.mean()))
        history["reward_max"].append(float(rewards_raw.max()))
        history["validity"].append(float(validity))
        history["kl_penalty"].append(float(kl_penalties.mean()))
        history["aug_reward_mean"].append(float(aug_t.mean()))

        mlflow.log_metrics({
            "reward_mean":     float(rewards_raw.mean()),
            "reward_max":      float(rewards_raw.max()),
            "validity":        float(validity),
            "kl_penalty_mean": float(kl_penalties.mean()),
            "aug_reward_mean": float(aug_t.mean()),
            "pg_loss":         float(loss.item()),
            "baseline":        float(baseline),
        }, step=step)

        # Store valid molecules generated this step
        for s, r in zip(smiles_list, rewards_raw):
            if Chem.MolFromSmiles(s):
                rew = compute_reward(s)
                rew.update({"step": step, "smiles": s})
                all_molecules.append(rew)

        if (step + 1) % 50 == 0:
            best_pkd = max(
                (compute_reward(s)["pred_pkd"] for s in smiles_list if Chem.MolFromSmiles(s)),
                default=float("nan"),
            )
            print(f"  Step {step+1:3d}  reward={rewards_raw.mean():.3f}  "
                  f"validity={validity:.2f}  kl={kl_penalties.mean():.3f}  "
                  f"best_pkd={best_pkd:.2f}")
            mlflow.log_metric("best_pkd_step", best_pkd, step=step)

    # ── Summary metrics at end of run ─────────────────────────────────────
    df_valid = pd.DataFrame([m for m in all_molecules if m.get("valid")])
    if not df_valid.empty:
        mlflow.log_metrics({
            "total_valid":        int(df_valid.shape[0]),
            "validity_rate":      float(df_valid.shape[0] / (RL_STEPS * RL_BATCH)),
            "best_reward_total":  float(df_valid["reward_total"].max()),
            "best_pred_pkd":      float(df_valid["pred_pkd"].max()),
            "mean_reward_final":  float(np.mean(history["reward_mean"][-10:])),
        })

    RL_RUN_ID = rl_run.info.run_id

print(f"\nRL complete. Total valid molecules collected: "
      f"{sum(1 for m in all_molecules if m['valid'])}")
print(f"MLflow run ID: {RL_RUN_ID}")

## 9. Results Analysis

After training, I summarise the RL behaviour with a compact Nature-style 2×2 figure. The panels focus on reward dynamics, policy stability, and the quality of the highest-scoring molecules without relying on presentation-style decoration.


In [9]:
# Rebuild plotting tables if the notebook is opened without rerunning RL
results_cache = RL_DIR / "rl_results.json"
payload = None

if ("history" not in globals()) or (len(history.get("step", [])) == 0):
    assert results_cache.exists(), f"Results cache not found at {results_cache}"
    payload = json.loads(results_cache.read_text())
    history = payload["history"]

if ("all_molecules" in globals()) and len(all_molecules) > 0:
    df_mols = pd.DataFrame(all_molecules)
else:
    if payload is None:
        assert results_cache.exists(), f"Results cache not found at {results_cache}"
        payload = json.loads(results_cache.read_text())
    df_mols = pd.DataFrame(payload.get("top_molecules", []))
    if not df_mols.empty and "valid" not in df_mols:
        df_mols["valid"] = True

if ("df_top" in globals()) and len(df_top) > 0:
    df_top = df_top.copy()
else:
    if payload is None:
        assert results_cache.exists(), f"Results cache not found at {results_cache}"
        payload = json.loads(results_cache.read_text())
    df_top = pd.DataFrame(payload.get("top_molecules", []))

if not df_top.empty:
    df_top = df_top.sort_values("reward_total", ascending=False).drop_duplicates("smiles").head(20)

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#222222",
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

fig, axes = plt.subplots(2, 2, figsize=(7.6, 5.8))
fig.patch.set_facecolor("white")
steps = np.asarray(history["step"])

# ── Panel a: reward dynamics ──────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(steps, history["reward_mean"], color="#4C78A8", lw=1.6, label="mean")
ax.plot(steps, history["reward_max"], color="#72B7B2", lw=1.4, label="max")
ax.set_title("Reward")
ax.set_xlabel("Step")
ax.set_ylabel("Reward")
ax.set_ylim(0, 1)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="lower right", fontsize=8)
stats_a = (
    f"n = {len(steps)}\n"
    f"final mean = {history['reward_mean'][-1]:.3f}\n"
    f"best max = {np.max(history['reward_max']):.3f}"
)
ax.text(
    0.98, 0.96, stats_a,
    transform=ax.transAxes, ha="right", va="top",
    fontsize=8,
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#BBBBBB", linewidth=0.6),
)
ax.text(-0.12, 1.10, "a)", transform=ax.transAxes, ha="left", va="bottom", fontsize=10, fontweight="bold", color="#222222")

# ── Panel b: validity and KL penalty ─────────────────────────────────────
ax = axes[0, 1]
ax.plot(steps, history["validity"], color="#54A24B", lw=1.6)
ax.set_title("Validity / KL")
ax.set_xlabel("Step")
ax.set_ylabel("Validity")
ax.set_ylim(0, 1.05)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax2 = ax.twinx()
ax2.plot(steps, history["kl_penalty"], color="#B279A2", lw=1.2, linestyle="--")
ax2.set_ylabel("KL penalty")
ax2.tick_params(axis="y", labelsize=8, colors="#555555")
ax2.spines["top"].set_visible(False)
validity_final = history["validity"][-1] if history["validity"] else float("nan")
kl_mean = float(np.mean(history["kl_penalty"])) if history["kl_penalty"] else float("nan")
stats_b = (
    f"n = {len(steps)}\n"
    f"final validity = {validity_final:.2f}\n"
    f"mean KL = {kl_mean:.3f}"
)
ax.text(
    0.98, 0.96, stats_b,
    transform=ax.transAxes, ha="right", va="top",
    fontsize=8,
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#BBBBBB", linewidth=0.6),
)
ax.text(-0.12, 1.10, "b)", transform=ax.transAxes, ha="left", va="bottom", fontsize=10, fontweight="bold", color="#222222")

# ── Panel c: top molecules, QED vs predicted pKd ────────────────────────
ax = axes[1, 0]
ax.scatter(df_top["r_qed"], df_top["pred_pkd"], s=34, color="#4C78A8", edgecolors="white", linewidths=0.5, alpha=0.9)
ax.set_title("Top molecules")
ax.set_xlabel("QED")
ax.set_ylabel("Predicted pKd")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
stats_c = (
    f"n = {len(df_top)}\n"
    f"best reward = {df_top['reward_total'].max():.3f}\n"
    f"best pKd = {df_top['pred_pkd'].max():.2f}"
)
ax.text(
    0.98, 0.96, stats_c,
    transform=ax.transAxes, ha="right", va="top",
    fontsize=8,
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#BBBBBB", linewidth=0.6),
)
ax.text(-0.12, 1.10, "c)", transform=ax.transAxes, ha="left", va="bottom", fontsize=10, fontweight="bold", color="#222222")

# ── Panel d: oracle pKd distribution ─────────────────────────────────────
ax = axes[1, 1]
pkd_values = df_mols["pred_pkd"].dropna().to_numpy()
bins = min(12, max(5, int(np.sqrt(len(pkd_values))))) if len(pkd_values) else 5
ax.hist(pkd_values, bins=bins, color="#72B7B2", edgecolor="white", linewidth=0.6)
ax.set_title("Oracle pKd")
ax.set_xlabel("Predicted pKd")
ax.set_ylabel("Count")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
dist_label = "all scored" if len(df_mols) > len(df_top) else "top set"
stats_d = (
    f"n = {len(pkd_values)} ({dist_label})\n"
    f"median = {np.median(pkd_values):.2f}\n"
    f"max = {np.max(pkd_values):.2f}"
)
ax.text(
    0.98, 0.96, stats_d,
    transform=ax.transAxes, ha="right", va="top",
    fontsize=8,
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#BBBBBB", linewidth=0.6),
)
ax.text(-0.12, 1.10, "d)", transform=ax.transAxes, ha="left", va="bottom", fontsize=10, fontweight="bold", color="#222222")

fig.subplots_adjust(left=0.10, right=0.95, bottom=0.11, top=0.90, wspace=0.32, hspace=0.48)
plt.savefig(str(RL_DIR / "phase3_results.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"Figure saved → {RL_DIR / 'phase3_results.png'}")

# ── Summary table ─────────────────────────────────────────────────────────
print("\n── Top 10 Generated Molecules ────────────────────────────────────")
print(df_top[["smiles","reward_total","pred_pkd","r_qed","r_sa","r_mw"]]
      .head(10).to_string(index=False))


Figure saved → /workspace/data/rl_results/phase3_results.png

── Top 10 Generated Molecules ────────────────────────────────────
                                                     smiles  reward_total  pred_pkd    r_qed     r_sa  r_mw
                           O=C(Nc1ccccc1)c1cccc(C(F)(F)F)c1      0.718876  7.469193 0.874520 0.852560   1.0
             NS(=O)(=O)c1ccc(C(=O)N2CCC(O)(c3ccccc3)CC2)cc1      0.705207  7.462093 0.865110 0.795404   1.0
            Cc1ccc(C(=O)NC2C=C2)cc1-c1ccc(C(=O)NCC2C=C2)cc1      0.676717  7.411624 0.790533 0.740146   1.0
                            Cc1ccc(C(=O)NCCCCCCC(=O)N=O)cc1      0.653566  7.556640 0.585520 0.793149   1.0
                                       O=C(O)C(=O)Cc1ccccc1      0.651119  6.973795 0.674542 0.837604   1.0
          Nc1nc(Nc2ccc(S(N)(=O)=O)cc2)nn1C(=O)c1c(F)cccc1Cl      0.638686  7.382211 0.593050 0.754827   1.0
O=C(Nn1c(=O)c(=O)[nH]c2cc(C(F)(F)F)c(-n3ccnc3)cc21)c1ccccc1      0.613131  7.365231 0.501449 0.722899   1.0
       

## 10. Save Results and Export `src/rl/`

The final step writes out the generated-molecule summaries and promotes the reusable policy code into `src/rl/`. That keeps the experimental notebook and the project source tree aligned.

In [ ]:
import json as _json

# ── Serialise results to JSON ─────────────────────────────────────────────
results_payload = {
    "experiment": "phase3_reinforce",
    "config": {
        "rl_steps": RL_STEPS, "rl_batch": RL_BATCH, "kl_beta": KL_BETA,
        "reward_weights": R_WEIGHTS, "prior_epochs": PRIOR_EPOCHS,
        "ref_pocket": ref_pdb_id, "ref_pkd": float(ref_graph.y_affinity.item()),
    },
    "history": history,
    "top_molecules": (
        df_top[["smiles","reward_total","pred_pkd","r_affinity","r_qed","r_sa","r_mw"]]
        .head(50).to_dict(orient="records")
    ),
    "summary": {
        "total_generated":   len(all_molecules),
        "total_valid":       int(df_mols["valid"].sum()),
        "validity_rate":     float(df_mols["valid"].mean()),
        "best_reward":       float(df_mols["reward_total"].max()),
        "best_pkd":          float(df_mols["pred_pkd"].max()),
        "mean_reward_final": float(np.mean(history["reward_mean"][-10:])),
    },
}

out_json = RL_DIR / "rl_results.json"
out_json.write_text(_json.dumps(results_payload, indent=2, default=str))
print(f"Results saved → {out_json}")
print(f"\nSummary:")
for k, v in results_payload["summary"].items():
    print(f"  {k:25s}: {v}")

# ── Save policy checkpoint ────────────────────────────────────────────────
rl_ckpt = ROOT / "checkpoints" / "policy_rl.pt"
torch.save({
    "policy_state_dict": policy.state_dict(),
    "tokenizer_vocab":   tokenizer.vocab,
    "config": {"hidden": POLICY_HIDDEN, "layers": POLICY_LAYERS,
               "embed_dim": EMBED_DIM},
}, rl_ckpt)
print(f"Policy saved   → {rl_ckpt}")

# ── Log artifacts to MLflow ───────────────────────────────────────────────
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
with mlflow.start_run(run_id=RL_RUN_ID):
    mlflow.log_artifact(str(out_json),              artifact_path="results")
    mlflow.log_artifact(str(rl_ckpt),               artifact_path="checkpoints")
    mlflow.log_artifact(str(RL_DIR / "phase3_results.png"), artifact_path="figures")
    # Log top-10 SMILES as a plain-text table for easy inspection in the UI
    top10_txt = RL_DIR / "top10_smiles.txt"
    top10_txt.write_text(
        df_top[["smiles","reward_total","pred_pkd","r_qed"]]
        .head(10).to_string(index=False)
    )
    mlflow.log_artifact(str(top10_txt), artifact_path="results")

# ── Export src/rl/ module ─────────────────────────────────────────────────
rl_dir = ROOT / "src" / "rl"
rl_dir.mkdir(parents=True, exist_ok=True)
(rl_dir / "__init__.py").write_text("")

(rl_dir / "policy.py").write_text("""\"\"\"SMILES policy loader for GNNBindOptimizer RL agent.\"\"\"
import torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path

class SMILESTokenizer:
    TWO_CHAR = {"Cl","Br","Si","Se","Na","Li","Mg","Ca","Fe","Cu","Zn","Al","As","@@"}
    PAD,BOS,EOS,UNK = "<PAD>","<BOS>","<EOS>","<UNK>"
    def __init__(self, vocab):
        self.vocab = vocab
        self.c2i = {c:i for i,c in enumerate(vocab)}
        self.i2c = {i:c for i,c in enumerate(vocab)}
        self.pad_idx = self.c2i[self.PAD]; self.bos_idx = self.c2i[self.BOS]
        self.eos_idx = self.c2i[self.EOS]; self.unk_idx = self.c2i[self.UNK]
    def tokenize(self, s):
        tokens, i = [], 0
        while i < len(s):
            two = s[i:i+2]
            if two in self.TWO_CHAR: tokens.append(two); i += 2
            else: tokens.append(s[i]); i += 1
        return tokens
    def encode(self, s):
        return [self.bos_idx] + [self.c2i.get(t, self.unk_idx) for t in self.tokenize(s)] + [self.eos_idx]
    def decode(self, ids):
        out = []
        for i in ids:
            if i == self.eos_idx: break
            if i in (self.pad_idx, self.bos_idx): continue
            out.append(self.i2c.get(i,"?"))
        return "".join(out)
    def __len__(self): return len(self.vocab)

class SMILESPolicy(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers):
        super().__init__()
        self.hidden_dim = hidden_dim; self.n_layers = n_layers
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True,
                             dropout=0.1 if n_layers > 1 else 0.0)
        self.fc    = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.embed(x), hidden)
        return self.fc(out), hidden

def load_policy(ckpt_path: Path, device="cpu"):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    tok  = SMILESTokenizer(ckpt["tokenizer_vocab"])
    cfg  = ckpt["config"]
    pol  = SMILESPolicy(len(tok), cfg["embed_dim"], cfg["hidden"], cfg["layers"]).to(device)
    pol.load_state_dict(ckpt["policy_state_dict"])
    pol.eval()
    return pol, tok
""")
print(f"src/rl/policy.py written")
print("\n── Part 3 complete ──────────────────────────────────────────────")
print("  → Part 4: SQL Server schema (db/init.sql)")